In [5]:
import subprocess
result = subprocess.run(['find', '/notebooks', '-name', '*.xlsx'], capture_output=True, text=True)
print(result.stdout)

In [7]:
"""
build_codebook_excel.py
PP422 | LSE Growth Co-Lab
Builds the variable codebook Excel from scratch using codebook.md as source of truth.
Output: PP422_Variable_Codebook.xlsx
"""

import pandas as pd
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# ── CODEBOOK DATA ─────────────────────────────────────────────────────────────
# Each dict = one row. Fields: variable, label, role, group, definition, source, notes

ROWS = [

    # ── DEPENDENT VARIABLES ──────────────────────────────────────────────────

    dict(
        variable="lq_emp",
        label="Employment Location Quotient",
        role="Dependent – primary",
        group="Dependent Variables",
        definition="Relative specialisation of a LAD in an IS8 sector based on employment.",
        measurement="(IS8 emp in LAD / total emp in LAD) / (IS8 emp nationally / total emp nationally). Values >1 = above-average specialisation.",
        source="Employee counts LAD (BRES via Nomis)",
        notes="Threshold = 1. Winsorise at p99 for Life Sciences, Advanced Manufacturing, Financial Services before regression.",
    ),
    dict(
        variable="lq_bus",
        label="Business Count Location Quotient",
        role="Dependent – robustness check",
        group="Dependent Variables",
        definition="Relative specialisation of a LAD in an IS8 sector based on business count.",
        measurement="(IS8 businesses in LAD / total businesses in LAD) / (IS8 businesses nationally / total businesses nationally).",
        source="Business counts LAD (UK Business Counts via Nomis)",
        notes="Winsorise at p99 for Life Sciences, Financial Services before regression.",
    ),
    dict(
        variable="gd_emp",
        label="Employment Growth Differential",
        role="Dependent – primary",
        group="Dependent Variables",
        definition="LAD IS8 employment trajectory minus national IS8 employment trajectory.",
        measurement="β_LAD_emp − β_national_emp. β estimated via log-linear OLS: log(y_t) ~ α + β·t. Units: log points per year ≈ pp per year.",
        source="Employee counts LAD (BRES via Nomis)",
        notes="Window 2016–2022. NaN if < 4 valid years (~13% null rate). Winsorise both tails for Life Sciences; monitor left tail for Financial Services, Advanced Manufacturing.",
    ),
    dict(
        variable="gd_bus",
        label="Business Count Growth Differential",
        role="Dependent – robustness check",
        group="Dependent Variables",
        definition="LAD IS8 business count trajectory minus national IS8 business count trajectory.",
        measurement="β_LAD_bus − β_national_bus. Same OLS method as gd_emp.",
        source="Business counts LAD (UK Business Counts via Nomis)",
        notes="Window 2016–2022. NaN if < 4 valid years (~17.5% null rate). Winsorise both tails for Life Sciences.",
    ),

    # ── DIAGNOSTIC (not used in analysis) ────────────────────────────────────

    dict(
        variable="n_years_emp",
        label="Valid Employment Years",
        role="Diagnostic – not used in analysis",
        group="Dependent Variables",
        definition="Count of non-zero, finite employment observations used in LAD slope estimate.",
        measurement="Integer count. GD set to NaN when < 4.",
        source="Employee counts LAD (BRES)",
        notes="Diagnostic column only.",
    ),
    dict(
        variable="n_years_bus",
        label="Valid Business Count Years",
        role="Diagnostic – not used in analysis",
        group="Dependent Variables",
        definition="Count of non-zero, finite business count observations used in LAD slope estimate.",
        measurement="Integer count. GD set to NaN when < 4.",
        source="Business counts LAD (UK Business Counts)",
        notes="Diagnostic column only.",
    ),
    dict(
        variable="emp_share",
        label="IS8 Employment Share",
        role="Diagnostic – not used in analysis",
        group="Dependent Variables",
        definition="IS8 sector employment as a share of total local employment. Used internally for LQ computation.",
        measurement="Percentage: IS8 employees / total employees in LAD × 100.",
        source="Employee counts LAD (BRES)",
        notes="Diagnostic column only. Complement to LQ.",
    ),

    # ── HUMAN CAPITAL ─────────────────────────────────────────────────────────

    dict(
        variable="nvq_level3",
        label="NVQ Level 3+ Qualifications",
        role="Independent – human capital",
        group="Human Capital",
        definition="Share of working-age residents holding qualifications at NVQ Level 3 or above (A-level equivalent or higher).",
        measurement="Percentage of working-age population (16–64). ONS snapshot ~2021.",
        source="ONS / NOMIS 2021",
        notes="Proxy for skilled labour stock available to IS8 firms. 0.7% null rate (City of London, Isles of Scilly).",
    ),
    dict(
        variable="gcse_age19",
        label="GCSEs by Age 19",
        role="Independent – human capital",
        group="Human Capital",
        definition="Share of young people achieving a standard pass (grade 4+) in English and Maths GCSEs by age 19.",
        measurement="Percentage. County/UA level — used at LAD level with caution.",
        source="ONS / DfE 2021/22",
        notes="Educational pipeline proxy. Geographic resolution below LAD — treat as approximate.",
    ),
    dict(
        variable="apprenticeship_starts",
        label="Apprenticeship Starts",
        role="Independent – human capital",
        group="Human Capital",
        definition="Number of apprenticeship starts in the area per period.",
        measurement="Count (or rate per working-age population where normalised). Annual.",
        source="ONS / DfE 2022/23",
        notes="Vocational training pipeline — particularly relevant to Advanced Manufacturing and Life Sciences.",
    ),
    dict(
        variable="apprenticeship_achievements",
        label="Apprenticeship Achievements",
        role="Independent – human capital",
        group="Human Capital",
        definition="Number of apprenticeships successfully completed in the area.",
        measurement="Count (or rate). Annual.",
        source="ONS / DfE 2022/23",
        notes="Measures stock of completed vocational training — workforce capability signal.",
    ),
    dict(
        variable="fe_participation",
        label="FE and Skills Participation",
        role="Independent – human capital",
        group="Human Capital",
        definition="Rate of participation in Further Education and skills programmes among working-age adults.",
        measurement="Percentage of working-age population. Annual.",
        source="ONS / DfE 2022/23",
        notes="Captures ongoing upskilling in the working-age population.",
    ),

    # ── ENTREPRENEURIAL DISCOVERY ─────────────────────────────────────────────

    dict(
        variable="enterprise_birth_rate",
        label="Enterprise Birth Rate",
        role="Independent – entrepreneurial discovery",
        group="Entrepreneurial Discovery",
        definition="New firm formation as a share of the active enterprise stock.",
        measurement="enterprise_births / enterprise_active. Normalised rate — removes LAD size effect.",
        source="ONS Business Demography 2022",
        notes="Proxy for entrepreneurial dynamism (EEG: entrepreneurial discovery). Moderately right-skewed — consider log transformation.",
    ),
    dict(
        variable="enterprise_death_rate",
        label="Enterprise Death Rate",
        role="Independent – entrepreneurial discovery",
        group="Entrepreneurial Discovery",
        definition="Businesses ceasing to trade as a share of the active enterprise stock.",
        measurement="enterprise_deaths / enterprise_active. Normalised rate.",
        source="ONS Business Demography 2022",
        notes="Schumpeterian churn signal. Directional ambiguity: creative destruction (positive) vs ecosystem fragility (negative). Moderately right-skewed — consider log transformation.",
    ),
    dict(
        variable="enterprise_high_growth_rate",
        label="High-Growth Enterprise Rate",
        role="Independent – entrepreneurial discovery",
        group="Entrepreneurial Discovery",
        definition="Share of active firms classified as high-growth (20%+ employment/turnover growth p.a. over 3 years).",
        measurement="enterprise_high_growth / enterprise_active. Normalised rate.",
        source="ONS Business Demography 2022",
        notes="Indicates depth of IS8 ecosystem — scaling-firm presence. Very low variance: mean 0.004, max 0.018 — likely shrunk by regularisation.",
    ),

    # ── CONNECTIVITY & ACCESSIBILITY ─────────────────────────────────────────

    dict(
        variable="transport_to_employer",
        label="Public Transport Access to Employment",
        role="Independent – connectivity",
        group="Connectivity & Accessibility",
        definition="Share of working-age residents who can access a defined number of employment sites within a set travel time by public transport.",
        measurement="Percentage. DfT snapshot 2019, referenced to 2011 boundaries.",
        source="DfT 2019",
        notes="Most temporally stale X variable — pre-COVID patterns may not reflect current accessibility.",
    ),
    dict(
        variable="drive_to_employer",
        label="Drive-Time Access to Employment",
        role="Independent – connectivity",
        group="Connectivity & Accessibility",
        definition="Share of working-age residents who can access employment sites within a set drive time.",
        measurement="Percentage. DfT snapshot 2019.",
        source="DfT 2019",
        notes="Relevant in car-dependent regions. Temporally stale — see transport_to_employer.",
    ),
    dict(
        variable="cycle_to_employer",
        label="Cycling Access to Employment",
        role="Independent – connectivity",
        group="Connectivity & Accessibility",
        definition="Share of working-age residents able to cycle to employment within a defined time/distance.",
        measurement="Percentage. DfT snapshot 2019.",
        source="DfT 2019",
        notes="Urban density signal — high values indicate compact, accessible urban labour markets.",
    ),
    dict(
        variable="broadband",
        label="Gigabit Broadband Availability",
        role="Independent – connectivity",
        group="Connectivity & Accessibility",
        definition="Share of premises with access to gigabit-capable broadband.",
        measurement="Percentage of premises.",
        source="Ofcom / ONS Sep 2023",
        notes="Digital infrastructure prerequisite for Digital & Technologies and Financial Services.",
    ),
    dict(
        variable="coverage_4g",
        label="4G Area Coverage",
        role="Independent – connectivity",
        group="Connectivity & Accessibility",
        definition="Share of geographic area with outdoor 4G mobile coverage from at least one operator.",
        measurement="Percentage of area.",
        source="Ofcom / ONS Sep 2023",
        notes="Baseline mobile digital infrastructure. Very low variance: mean 99.5% — likely shrunk by regularisation.",
    ),

    # ── LABOUR MARKET CONDITIONS ──────────────────────────────────────────────

    dict(
        variable="unemployment_rate",
        label="Unemployment Rate",
        role="Independent – labour market",
        group="Labour Market Conditions",
        definition="Share of economically active residents who are unemployed and seeking work.",
        measurement="Percentage. Annual model-based estimate for LADs.",
        source="ONS / NOMIS 2022/23",
        notes="Directional ambiguity: labour flexibility (positive) vs economic weakness (negative). 5.7% null rate (17 LADs — suppressed small-area estimates). Do not impute.",
    ),

    # ── PLACE CONDITIONS ──────────────────────────────────────────────────────

    dict(
        variable="housing_net_additions",
        label="Net Additions to Housing Stock",
        role="Independent – place conditions",
        group="Place Conditions",
        definition="Net annual additions to the local housing stock (new builds minus demolitions/conversions).",
        measurement="Count of dwellings per year.",
        source="DLUHC / MHCLG FY2023",
        notes="Housing supply capacity — affects ability to attract and retain workers for IS8 sectors.",
    ),

    # ── DERIVED VARIABLES ─────────────────────────────────────────────────────

    dict(
        variable="related_variety",
        label="Related Variety",
        role="Independent – derived (EEG)",
        group="Derived Variables",
        definition="Degree to which a LAD's industrial mix contains sectors that are technologically or capability-adjacent to IS8 sectors.",
        measurement="Shannon entropy across IS8 sectors per LAD × year, weighted by business counts. Higher = more diversified. Theoretical max = log(6) ≈ 1.79; observed max = 1.35.",
        source="Business counts LAD (computed in IndicatorBuilder)",
        notes="Core EEG concept. Approximation only — true EEG related variety requires SIC-level proximity weights. LAD × year level (same value across all sectors within a LAD × year).",
    ),
    dict(
        variable="within_sector_diversity",
        label="Within-Sector Diversity",
        role="Independent – derived (EEG)",
        group="Derived Variables",
        definition="Shannon entropy across SIC codes within each IS8 sector per LAD × year. Measures internal sectoral complexity.",
        measurement="Shannon entropy across SIC codes within IS8 sector per LAD × year.",
        source="Business counts LAD (computed in IndicatorBuilder)",
        notes="Complements related_variety. Measures internal complexity rather than cross-sector diversity.",
    ),
    dict(
        variable="size_large_share",
        label="Large Firm Share",
        role="Independent – derived (firm structure)",
        group="Derived Variables",
        definition="Share of local IS8 businesses with 250+ employees.",
        measurement="Percentage of IS8 business units in LAD. Computed in IndicatorBuilder.",
        source="Business counts LAD (by size band)",
        notes="Captures presence of anchor employers — relevant to IS8 ecosystem depth. Very low variance: mean 0.004, max 0.125 — monitor in regularised regression.",
    ),
    dict(
        variable="size_micro_share",
        label="Micro Firm Share",
        role="Independent – derived (firm structure)",
        group="Derived Variables",
        definition="Share of local IS8 businesses with 0–9 employees.",
        measurement="Percentage of IS8 business units in LAD. Computed in IndicatorBuilder.",
        source="Business counts LAD (by size band)",
        notes="High micro share may indicate entrepreneurial dynamism or thin market — interpret alongside enterprise_high_growth_rate.",
    ),

    # ── EXCLUDED VARIABLES ────────────────────────────────────────────────────

    dict(
        variable="gva_per_hour",
        label="GVA per Hour Worked",
        role="EXCLUDED",
        group="Excluded Variables",
        definition="Gross value added per hour worked — local productivity measure.",
        measurement="£ per hour.",
        source="ONS Regional Accounts",
        notes="Excluded: reverse causality — IS8 cluster presence raises local productivity; outcome, not precondition.",
    ),
    dict(
        variable="weekly_pay",
        label="Gross Median Weekly Pay",
        role="EXCLUDED",
        group="Excluded Variables",
        definition="Median gross weekly earnings of full-time workers.",
        measurement="£ per week.",
        source="ONS ASHE",
        notes="Excluded: IS8 firms pay above-average wages — outcome of sectoral composition, not enabler.",
    ),
    dict(
        variable="gdhi_per_head",
        label="GDHI per Head",
        role="EXCLUDED",
        group="Excluded Variables",
        definition="Gross Disposable Household Income per capita.",
        measurement="£ per person.",
        source="ONS Regional Accounts",
        notes="Excluded: reflects sectoral composition of local economy — outcome, not precondition.",
    ),
    dict(
        variable="employment_rate",
        label="Employment Rate (16–64)",
        role="EXCLUDED",
        group="Excluded Variables",
        definition="Share of working-age (16–64) population in employment.",
        measurement="Percentage.",
        source="ONS model-based estimates",
        notes="Excluded: general economic health indicator — likely outcome of IS8 presence; redundant with unemployment_rate.",
    ),
]

# ── BUILD DATAFRAME ───────────────────────────────────────────────────────────

COLUMNS = ["variable", "label", "role", "group", "definition", "measurement", "source", "notes"]

df = pd.DataFrame(ROWS, columns=COLUMNS)

# ── STYLING HELPERS ───────────────────────────────────────────────────────────

# Colour palette
NAVY      = "1F3864"
LIGHT_ROW = "EBF1FF"
WHITE     = "FFFFFF"
EXCL_FILL = "FFF2CC"   # amber tint for excluded rows
DIAG_FILL = "F2F2F2"   # grey tint for diagnostic rows

GROUP_COLORS = {
    "Dependent Variables":       "D6E4F0",
    "Human Capital":             "D5E8D4",
    "Entrepreneurial Discovery": "FFE6CC",
    "Connectivity & Accessibility": "E1D5E7",
    "Labour Market Conditions":  "FFF2CC",
    "Place Conditions":          "DAE8FC",
    "Derived Variables":         "F8CECC",
    "Excluded Variables":        "F2F2F2",
}

def make_fill(hex_color):
    return PatternFill("solid", fgColor=hex_color)

def thin_border():
    s = Side(style="thin", color="CCCCCC")
    return Border(left=s, right=s, top=s, bottom=s)

# ── WRITE EXCEL ───────────────────────────────────────────────────────────────

OUTPUT_FILE = "PP422_Variable_Codebook.xlsx"

with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
    df.to_excel(writer, index=False, sheet_name="Codebook")
    ws = writer.sheets["Codebook"]

    # ── Header row ────────────────────────────────────────────────────────────
    hdr_fill = make_fill(NAVY)
    hdr_font = Font(bold=True, color="FFFFFF", size=11, name="Calibri")
    for cell in ws[1]:
        cell.fill      = hdr_fill
        cell.font      = hdr_font
        cell.alignment = Alignment(horizontal="center", vertical="center",
                                   wrap_text=True)
        cell.border    = thin_border()
    ws.row_dimensions[1].height = 30

    # ── Data rows ─────────────────────────────────────────────────────────────
    group_col = COLUMNS.index("group") + 1   # 1-based
    role_col  = COLUMNS.index("role")  + 1

    for row_idx, row in enumerate(ws.iter_rows(min_row=2), start=2):
        grp  = row[group_col - 1].value or ""
        role = row[role_col  - 1].value or ""

        # Pick fill
        fill_hex = GROUP_COLORS.get(grp, WHITE)
        row_fill = make_fill(fill_hex)

        for cell in row:
            cell.fill      = row_fill
            cell.font      = Font(size=10, name="Calibri")
            cell.alignment = Alignment(wrap_text=True, vertical="top")
            cell.border    = thin_border()

        ws.row_dimensions[row_idx].height = 60

    # ── Column widths ─────────────────────────────────────────────────────────
    COL_WIDTHS = {
        "variable":    18,
        "label":       28,
        "role":        28,
        "group":       24,
        "definition":  50,
        "measurement": 50,
        "source":      30,
        "notes":       55,
    }
    for col_idx, col_name in enumerate(COLUMNS, start=1):
        ws.column_dimensions[get_column_letter(col_idx)].width = COL_WIDTHS[col_name]

    # ── Freeze header ─────────────────────────────────────────────────────────
    ws.freeze_panes = "A2"

    # ── Auto-filter ───────────────────────────────────────────────────────────
    ws.auto_filter.ref = ws.dimensions

print(f"✓ Saved {len(df)} variables to '{OUTPUT_FILE}'")
print(f"  Groups: {df['group'].value_counts().to_dict()}")

✓ Saved 30 variables to 'PP422_Variable_Codebook.xlsx'
  Groups: {'Dependent Variables': 7, 'Human Capital': 5, 'Connectivity & Accessibility': 5, 'Derived Variables': 4, 'Excluded Variables': 4, 'Entrepreneurial Discovery': 3, 'Labour Market Conditions': 1, 'Place Conditions': 1}
